# Daily Demand Forecasting – End-to-End ML Mini Project
**Goal:** predict `total_orders` (daily order demand) from the other columns.
**Problem type:** *Regression* (the target is a continuous number, not a class).

Flow: Data Understanding → Cleaning → EDA → Correlation → Feature Engineering → Encoding → Split & Scale → Feature Selection → Models → Evaluation → Tuning → Final Model → Save → API → UI → Docker/Render

## 0. Imports

In [ ]:
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

# 1. Data Understanding

### 1.1 Load the dataset

In [ ]:
df = pd.read_csv("Daily_Demand_Forecasting_Orders.csv")
df.head()

### 1.2 Rename the columns (the original names are long and hard to type)

In [ ]:
df.columns = [
    "row_id", "week_of_month", "day_of_week",
    "non_urgent_orders", "urgent_orders",
    "order_type_a", "order_type_b", "order_type_c",
    "fiscal_sector_orders", "traffic_controller_orders",
    "banking_orders_1", "banking_orders_2", "banking_orders_3",
    "total_orders",
]
df.head()

### 1.3 Number of rows and columns

In [ ]:
print("Rows    :", df.shape[0])
print("Columns :", df.shape[1])

### 1.4 Data types and info

In [ ]:
df.info()

### 1.5 Identify numerical and categorical columns

In [ ]:
# week_of_month (1-5) and day_of_week (2-6) are stored as numbers but they are really CATEGORIES.
categorical_cols = ["week_of_month", "day_of_week"]
target_col = "total_orders"
numerical_cols = [c for c in df.columns if c not in categorical_cols + [target_col, "row_id"]]

print("Categorical columns:", categorical_cols)
print("Numerical columns  :", numerical_cols)
print("Target variable    :", target_col)
print("Ignored (just an index):", "row_id")

**Meaning of the fields**
- `week_of_month` 1–5, `day_of_week` 2–6 (2 = Monday … 6 = Friday)
- `non_urgent_orders`, `urgent_orders` – orders by urgency
- `order_type_a/b/c` – orders by type
- `fiscal_sector_orders`, `traffic_controller_orders` – orders from those sectors
- `banking_orders_1/2/3` – orders from three banks
- `total_orders` – **target**

*(The assignment mentions "churn / non-churn" because it is a generic template. Our target is a number, so there is no class label – we do regression instead of classification.)*

### 1.6 Descriptive statistics

In [ ]:
df.describe().T

### 1.7 Target distribution (the regression version of 'class distribution')

In [ ]:
print(df[target_col].describe())
print("Skewness:", round(df[target_col].skew(), 2))

plt.figure(figsize=(7, 4))
sns.histplot(df[target_col], kde=True, color="steelblue")
plt.title("Distribution of total_orders")
plt.show()

**Observation:** the target is right-skewed (skewness ≈ 1.4). Most days have 200–350 orders, a few busy days go above 500. There are only **60 rows**, so we must be careful about over-fitting.

# 2. Data Cleaning

### 2.1 Missing values  
*Why:* models like Linear Regression crash on NaN values and they hide information.

In [ ]:
df.isnull().sum()

**Observation:** no missing values, so nothing to impute.

### 2.2 Duplicate rows  
*Why:* duplicates over-weight some rows and can leak into the test set.

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)

### 2.3 Incorrect data types  
*Why:* `week_of_month` and `day_of_week` are numbers, but 5 > 2 has no real meaning for a day code. They must be treated as categories.

In [ ]:
day_map = {2: "Monday", 3: "Tuesday", 4: "Wednesday", 5: "Thursday", 6: "Friday"}
df["day_name"] = df["day_of_week"].map(day_map)
df[["day_of_week", "day_name"]].drop_duplicates().sort_values("day_of_week")

**Observation:** every other column is a proper number, so no conversion is needed. The categorical columns are encoded later in Section 6.

### 2.4 Outlier detection (IQR method)  
*Why:* extreme values can pull a model's line towards them.

In [ ]:
check_cols = numerical_cols + [target_col]
Q1 = df[check_cols].quantile(0.25)
Q3 = df[check_cols].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outlier_counts = ((df[check_cols] < lower) | (df[check_cols] > upper)).sum()
outlier_table = pd.DataFrame({
    "outliers": outlier_counts,
    "percent": (outlier_counts / len(df) * 100).round(1),
})
outlier_table

**Observation:** `fiscal_sector_orders` has the most outliers (13), then `banking_orders_1` (10).
**Decision:** we **keep** them. With only 60 rows deleting 20 % of the data is costly, and these spikes are real business days (e.g. month-end). Tree models are not affected by outliers; for linear models we log-transform `fiscal_sector_orders` in feature engineering.

### 2.5 Invalid values  
*Why:* negative orders or a week number of 9 would be data-entry errors.

In [ ]:
print("Negative values in numeric columns :", (df[check_cols] < 0).sum().sum())
print("week_of_month outside 1-5           :", (~df["week_of_month"].between(1, 5)).sum())
print("day_of_week outside 2-6             :", (~df["day_of_week"].between(2, 6)).sum())
print("Target <= 0                         :", (df[target_col] <= 0).sum())

**Observation:** all zero, so the data has no invalid values. (Zeros in `fiscal_sector_orders` are valid – some days had no fiscal orders.)

### 2.6 Inconsistent categorical values  
*Why:* 'Monday' and 'monday ' would be counted as two categories.

In [ ]:
print(df["week_of_month"].value_counts().sort_index())
print()
print(df["day_name"].value_counts())

**Observation:** only the expected categories (weeks 1–5, Monday–Friday) exist, with no spelling variants.

### 2.7 Unnecessary columns  
*Why:* an ID/index column carries no information and can confuse the model.

In [ ]:
df = df.drop(columns=["row_id"])
df.columns.tolist()

### 2.8 Leakage detection  
*Why:* if a feature is simply a re-statement of the target, the model looks perfect in testing but is useless in real life.

In [ ]:
diff = (df["order_type_a"] + df["order_type_b"] + df["order_type_c"] - df[target_col]).abs()
print("Max difference between (A + B + C) and total_orders:", diff.max())

**Observation:** `order_type_a + order_type_b + order_type_c` equals `total_orders` exactly, so the three columns are **target leakage**. We drop them. This step is the single most important cleaning decision in the project.

In [ ]:
df = df.drop(columns=["order_type_a", "order_type_b", "order_type_c"])
numerical_cols = [c for c in numerical_cols if not c.startswith("order_type")]
print("Remaining numerical columns:", numerical_cols)
df.head()

# 3. Exploratory Data Analysis (EDA)

### 3.1 Histograms of numerical features

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), numerical_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="teal")
    ax.set_title(col)
plt.tight_layout()
plt.show()

**Observation:** `fiscal_sector_orders` is extremely skewed (most days are close to 0, a few days are in the hundreds). `non_urgent_orders` and the banking columns are right-skewed. `traffic_controller_orders` is the most spread out.

### 3.2 Box plots (also outlier analysis)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), numerical_cols):
    sns.boxplot(y=df[col], ax=ax, color="salmon")
    ax.set_title(col)
plt.tight_layout()
plt.show()

**Observation:** the dots above the whiskers confirm the outliers counted in 2.4, mostly on the high side.

### 3.3 Count plots of categorical features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(x="week_of_month", data=df, palette="Set2", ax=axes[0])
axes[0].set_title("Rows per week of month")
sns.countplot(x="day_name", data=df, palette="Set1", ax=axes[1],
              order=["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"])
axes[1].set_title("Rows per weekday")
plt.tight_layout()
plt.show()

**Observation:** weekdays are almost perfectly balanced (11–13 rows each). Weeks 1 and 5 have fewer rows because those weeks are often partial.

### 3.4 Feature vs target – categorical features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x="day_name", y=target_col, data=df, palette="Set1", ax=axes[0],
            order=["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"])
axes[0].set_title("Total orders by weekday")
sns.boxplot(x="week_of_month", y=target_col, data=df, palette="Set2", ax=axes[1])
axes[1].set_title("Total orders by week of month")
plt.tight_layout()
plt.show()

print(df.groupby("day_name")[target_col].mean().round(0).sort_values(ascending=False))

**Observation:** **Monday is the busiest day (average ≈ 390 orders)** and demand falls through the week to ≈ 263 on Friday. Week of the month has a weaker effect; week 5 is highest (≈ 347) and week 2 the lowest (≈ 266).

### 3.5 Feature vs target – numerical features

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), numerical_cols):
    sns.regplot(x=df[col], y=df[target_col], ax=ax,
                scatter_kws={"alpha": 0.6}, line_kws={"color": "red"})
    ax.set_title(f"{col} vs total_orders")
plt.tight_layout()
plt.show()

**Observation:** `non_urgent_orders` has a very tight upward line, `banking_orders_2` and `urgent_orders` also rise clearly with demand. `fiscal_sector_orders` and `banking_orders_3` show almost no pattern.

# 4. Correlation Analysis

### 4.1 Correlation heatmap  
(Only continuous columns – the week/day codes are categories, so Pearson correlation on them would be misleading.)

In [ ]:
corr = df[numerical_cols + [target_col]].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation heatmap")
plt.show()

### 4.2 Correlation with the target

In [ ]:
corr[target_col].drop(target_col).sort_values(ascending=False)

### 4.3 Highly correlated feature pairs (multicollinearity check)

In [ ]:
pairs = corr.drop(index=target_col, columns=target_col).abs().unstack()
pairs = pairs[pairs < 1].sort_values(ascending=False)
pairs = pairs.iloc[::2]            # every pair appears twice, keep one
pairs.head(6)

**Observations**
- **Strong positive with target:** `non_urgent_orders` (0.93), `banking_orders_2` (0.80), `urgent_orders` (0.73), `banking_orders_1` (0.63).
- **Negative correlation:** none of the numeric features is meaningfully negative with the target (`day_of_week` is negative, but that is a category – handled in encoding).
- **Little or no correlation:** `fiscal_sector_orders` (−0.05), `banking_orders_3` (0.10).
- **Multicollinearity:** `non_urgent_orders` ↔ `banking_orders_2` (0.79) and ↔ `banking_orders_1` (0.73) are the highest pairs; none is above 0.8–0.9, so we **keep all** of them. Ridge regression and tree models handle moderate correlation well.

# 5. Feature Engineering

### 5.1 Urgent-order ratio  
*Why:* the *share* of urgent orders says more about a day's pressure than the raw count.

In [ ]:
df["urgent_ratio"] = df["urgent_orders"] / (df["urgent_orders"] + df["non_urgent_orders"])
df["urgent_ratio"].describe()

### 5.2 Total banking orders

In [ ]:
df["banking_total"] = df["banking_orders_1"] + df["banking_orders_2"] + df["banking_orders_3"]
df[["banking_total"]].describe()

### 5.3 Log-transform the skewed fiscal column  
*Why:* `log1p` squeezes the huge spikes so they stop dominating linear models (`log1p` handles zeros safely).

In [ ]:
df["fiscal_log"] = np.log1p(df["fiscal_sector_orders"])
print("Skew before:", round(df["fiscal_sector_orders"].skew(), 2))
print("Skew after :", round(df["fiscal_log"].skew(), 2))

### 5.4 Check the new features against the target

In [ ]:
new_features = ["urgent_ratio", "banking_total", "fiscal_log"]
df[new_features + [target_col]].corr()[target_col].drop(target_col)

**Observation:** `banking_total` (0.88) is much more useful than `banking_orders_3` alone. The original `fiscal_sector_orders` is replaced by its log version.

# 6. Encoding (separate step)
`week_of_month` and `day_name` are categorical, so they must become numbers. We use **One-Hot Encoding**: each category becomes its own 0/1 column. 
We use `drop_first=True` so the first category is the 'all zeros' baseline – this avoids the *dummy-variable trap* (perfect multicollinearity) in linear models.

### 6.1 Treat the codes as proper categories

In [ ]:
df["week_of_month"] = df["week_of_month"].astype(str)   # "1","2",...  -> text categories
# fix the order of the days so that Monday (not Friday) becomes the dropped baseline category
df["day_name"] = pd.Categorical(df["day_name"],
                                categories=["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"],
                                ordered=True)
df_model = df.drop(columns=["day_of_week", "fiscal_sector_orders"])   # day_name replaces the numeric day code
df_model.dtypes

### 6.2 One-hot encode

In [ ]:
df_encoded = pd.get_dummies(df_model, columns=["week_of_month", "day_name"], drop_first=True, dtype=int)
print(df_encoded.shape)
df_encoded.head()

### 6.3 Encoding table

In [ ]:
order_days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
week_cols = ["week_of_month_2", "week_of_month_3", "week_of_month_4", "week_of_month_5"]
day_cols = ["day_name_Tuesday", "day_name_Wednesday", "day_name_Thursday", "day_name_Friday"]

week_table = pd.DataFrame(
    [[w] + [int(c.split("_")[-1] == str(w)) for c in week_cols] for w in range(1, 6)],
    columns=["Original week_of_month"] + week_cols)

day_table = pd.DataFrame(
    [[d] + [int(c.split("_")[-1] == d) for c in day_cols] for d in order_days],
    columns=["Original day_name"] + day_cols)

display(week_table)
display(day_table)

**How to read it:** Week 1 and Monday are the *baseline* (all zeros). Week 3 → `week_of_month_3 = 1`, others 0. Thursday → `day_name_Thursday = 1`, others 0.

### 6.4 Verify the table against the real encoded data

In [ ]:
df_encoded[week_cols + day_cols].sum()

# 7. Train / Test Split  
*(Done before feature selection so the test set is never used to choose features.)*

In [ ]:
X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col]
print("Features:", X.shape[1])
print(list(X.columns))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("Train:", X_train.shape, " Test:", X_test.shape)

We use 80 / 20. With 60 rows the test set has only 12 rows, so we will also use **5-fold cross-validation** to get more reliable scores.

### 7.1 Scaling  
*Why:* Linear/Ridge regression is sensitive to feature size (orders in the hundreds vs ratios below 1). The scaler is **fit on the training data only** and then applied to the test data, otherwise test information leaks in.

In [ ]:
scale_cols = ["non_urgent_orders", "urgent_orders", "traffic_controller_orders",
              "banking_orders_1", "banking_orders_2", "banking_orders_3",
              "urgent_ratio", "banking_total", "fiscal_log"]

scaler = StandardScaler()
X_train_s = X_train.copy()
X_test_s = X_test.copy()
X_train_s[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_s[scale_cols] = scaler.transform(X_test[scale_cols])
X_train_s[scale_cols].describe().loc[["mean", "std"]].round(2)

# 8. Feature Selection

### 8.1 Mutual information

In [ ]:
mi = mutual_info_regression(X_train_s, y_train, random_state=42)
mi_series = pd.Series(mi, index=X_train_s.columns).sort_values(ascending=False)
mi_series.round(3)

### 8.2 Random Forest feature importance

In [ ]:
rf_sel = RandomForestRegressor(n_estimators=200, random_state=42)
rf_sel.fit(X_train_s, y_train)
imp = pd.Series(rf_sel.feature_importances_, index=X_train_s.columns).sort_values(ascending=False)
imp.round(3)

In [ ]:
imp.plot(kind="barh", figsize=(8, 6), color="seagreen")
plt.gca().invert_yaxis()
plt.title("Random Forest feature importance")
plt.show()

### 8.3 Pick the features

In [ ]:
selected = list(imp[imp >= 0.01].index)
print("Selected:", len(selected), "of", X_train_s.shape[1])
print(selected)

### 8.4 Performance before vs after feature selection (5-fold CV on the training set)

In [ ]:
def cv_r2(model, Xtr):
    return cross_val_score(model, Xtr, y_train, cv=5, scoring="r2").mean()

rows = []
for name, model in [("Linear Regression", LinearRegression()),
                    ("Random Forest", RandomForestRegressor(n_estimators=200, random_state=42))]:
    rows.append([name, cv_r2(model, X_train_s), cv_r2(model, X_train_s[selected])])

pd.DataFrame(rows, columns=["Model", "CV R2 - all features", "CV R2 - selected features"]).round(3)

**Observation:** with only the 8 selected features the CV score is the same or slightly better (Linear Regression 0.953 → 0.964, Random Forest 0.778 → 0.788). The 9 dropped features (fiscal, banking_orders_3 and most week/day columns) were only adding noise, so we continue with the smaller set.

In [ ]:
X_train_f = X_train_s[selected]
X_test_f = X_test_s[selected]
X_train_f.shape

# 9. Build Multiple Models

We train 4 models: Linear Regression, Ridge, Decision Tree and Random Forest.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
}

for name, m in models.items():
    m.fit(X_train_f, y_train)
    print("Trained:", name)

# 10. Model Evaluation
For regression the confusion matrix, precision, recall and ROC-AUC do **not** apply (they need classes). The regression equivalents are:
- **MAE** – average error in orders (easy to explain)
- **RMSE** – like MAE but punishes big mistakes
- **R²** – share of the variation the model explains (1 = perfect, 0 = no better than predicting the mean)
- **Actual-vs-Predicted plot** – replaces the confusion matrix

In [ ]:
def evaluate(model, X_tr, y_tr, X_te, y_te):
    pred_tr = model.predict(X_tr)
    pred_te = model.predict(X_te)
    return {
        "Train R2": r2_score(y_tr, pred_tr),
        "Test R2": r2_score(y_te, pred_te),
        "Test MAE": mean_absolute_error(y_te, pred_te),
        "Test RMSE": np.sqrt(mean_squared_error(y_te, pred_te)),
    }

results = {name: evaluate(m, X_train_f, y_train, X_test_f, y_test) for name, m in models.items()}
results_df = pd.DataFrame(results).T
results_df.round(3)

**Train vs test:** a big gap between Train R² and Test R² means over-fitting. Decision Tree and Random Forest usually show this on tiny data sets.

### 10.1 Cross-validated R² (more reliable than one 12-row test set)

In [ ]:
cv_scores = {name: cross_val_score(m, X_train_f, y_train, cv=5, scoring="r2").mean()
             for name, m in models.items()}
pd.Series(cv_scores, name="CV R2").round(3)

### 10.2 Actual vs predicted

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (name, m) in zip(axes, models.items()):
    pred = m.predict(X_test_f)
    ax.scatter(y_test, pred, color="royalblue")
    lims = [y_test.min() - 20, y_test.max() + 20]
    ax.plot(lims, lims, "r--")
    ax.set_title(name)
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
plt.tight_layout()
plt.show()

### 10.3 Residual plot for the best-looking model  
Residuals = actual − predicted. They should scatter randomly around 0.

In [ ]:
best_name = results_df["Test R2"].idxmax()
print("Best test R2:", best_name)
resid = y_test - models[best_name].predict(X_test_f)
plt.figure(figsize=(7, 4))
plt.scatter(models[best_name].predict(X_test_f), resid, color="darkorange")
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.title(f"Residuals - {best_name}")
plt.show()

# 11. Hyperparameter Tuning
We tune the best 2 families: **Ridge** (alpha) and **Random Forest**. Compare *before tuning → tuning → after tuning*.

### 11.1 Tune Ridge

In [ ]:
ridge_grid = GridSearchCV(
    Ridge(),
    param_grid={"alpha": [0.01, 0.1, 1, 3, 10, 30, 100]},
    cv=5, scoring="r2")
ridge_grid.fit(X_train_f, y_train)
print("Best params :", ridge_grid.best_params_)
print("Best CV R2  :", round(ridge_grid.best_score_, 3))

### 11.2 Tune Random Forest

In [ ]:
rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid={"n_estimators": [100, 200],
                "max_depth": [3, 5, None],
                "min_samples_leaf": [1, 3, 5]},
    cv=5, scoring="r2")
rf_grid.fit(X_train_f, y_train)
print("Best params :", rf_grid.best_params_)
print("Best CV R2  :", round(rf_grid.best_score_, 3))

### 11.3 Before vs after tuning

In [ ]:
tuned = {"Ridge (tuned)": ridge_grid.best_estimator_,
         "Random Forest (tuned)": rf_grid.best_estimator_}

compare = pd.DataFrame({name: evaluate(m, X_train_f, y_train, X_test_f, y_test)
                        for name, m in tuned.items()}).T
before = results_df.loc[["Ridge", "Random Forest"]]
before.index = ["Ridge (before)", "Random Forest (before)"]
pd.concat([before, compare]).round(3)

# 12. Final Model Selection
We do **not** just pick the highest score. We compare accuracy (R², MAE), over-fitting gap, inference time and complexity.

In [ ]:
final_candidates = {
    "Linear Regression": models["Linear Regression"],
    "Ridge (tuned)": ridge_grid.best_estimator_,
    "Decision Tree": models["Decision Tree"],
    "Random Forest (tuned)": rf_grid.best_estimator_,
}

rows = []
for name, m in final_candidates.items():
    r = evaluate(m, X_train_f, y_train, X_test_f, y_test)
    start = time.perf_counter()
    for _ in range(100):
        m.predict(X_test_f)
    ms = (time.perf_counter() - start) / 100 * 1000
    r["Overfit gap"] = r["Train R2"] - r["Test R2"]
    r["CV R2"] = cross_val_score(m, X_train_f, y_train, cv=5, scoring="r2").mean()
    r["Inference ms"] = ms
    rows.append(pd.Series(r, name=name))

final_table = pd.DataFrame(rows)
final_table.round(3)

**Decision (justify in your viva):** **Ridge (tuned, alpha = 0.1)** is selected for deployment.
- Best cross-validated R² (≈ 0.96) – far above Random Forest (≈ 0.79) and Decision Tree (≈ 0.69).
- Lowest test MAE (≈ 21 orders on an average day of ≈ 300) and the smallest over-fitting gap among the good models.
- Very small and fast (≈ 1 ms vs ≈ 16 ms for Random Forest) and easy to explain: one coefficient per feature.
- Ridge is preferred over plain Linear Regression because several inputs are correlated (e.g. non-urgent ↔ banking orders) and Ridge keeps the coefficients stable.
- The Decision Tree scored a perfect 1.0 on train data – a clear sign of memorising the data.
- Note: Test R² (≈ 0.78) is lower than CV R² (≈ 0.96) because the test set is only 12 rows and contains a few high-demand days; this is why we trust cross-validation more here.

# 13. Save the Model (joblib)  
We save the model **and** the scaler **and** the column order, because the API must apply exactly the same preprocessing.

In [ ]:
final_model = ridge_grid.best_estimator_   # <- change this if your table says a different model is best

bundle = {
    "model": final_model,
    "scaler": scaler,
    "scale_cols": scale_cols,
    "all_columns": list(X.columns),      # column order BEFORE feature selection
    "selected_features": selected,
    "model_version": "1.0.0",
    "test_mae": float(mean_absolute_error(y_test, final_model.predict(X_test_f))),
}
joblib.dump(bundle, "model.pkl")
print("Saved model.pkl")

### 13.1 Reload and test

In [ ]:
loaded = joblib.load("model.pkl")
print("Version:", loaded["model_version"])
print("Prediction on first test row :", loaded["model"].predict(X_test_f.iloc[[0]])[0].round(2))
print("Actual value                 :", y_test.iloc[0])

### 13.2 Build a sample input exactly like the API will receive

In [ ]:
sample = {
    "week_of_month": 3, "day_of_week": "Wednesday",
    "non_urgent_orders": 150.0, "urgent_orders": 110.0,
    "fiscal_sector_orders": 10.0, "traffic_controller_orders": 40000,
    "banking_orders_1": 50000, "banking_orders_2": 70000, "banking_orders_3": 20000,
}
print(sample)

In [ ]:
def prepare_input(data, bundle):
    row = pd.DataFrame([data])
    # feature engineering (same as Section 5)
    row["urgent_ratio"] = row["urgent_orders"] / (row["urgent_orders"] + row["non_urgent_orders"])
    row["banking_total"] = row["banking_orders_1"] + row["banking_orders_2"] + row["banking_orders_3"]
    row["fiscal_log"] = np.log1p(row["fiscal_sector_orders"])
    # encoding (same as Section 6) - build every dummy column, default 0
    full = pd.DataFrame(0, index=row.index, columns=bundle["all_columns"])
    for col in row.columns:
        if col in full.columns:
            full[col] = row[col]
    week_col = f"week_of_month_{data['week_of_month']}"
    day_col = f"day_name_{data['day_of_week']}"
    for c in (week_col, day_col):
        if c in full.columns:          # baseline category (week 1 / Monday) has no column -> stays all zeros
            full[c] = 1
    # scaling
    full[bundle["scale_cols"]] = bundle["scaler"].transform(full[bundle["scale_cols"]])
    return full[bundle["selected_features"]]

x_new = prepare_input(sample, loaded)
print("Predicted total orders:", round(loaded["model"].predict(x_new)[0], 1))

This `prepare_input` function is exactly what the FastAPI backend will use.

# 14. FastAPI Backend (files in the `backend/` folder)
Create the files in `backend/` (code given separately): `main.py`, `requirements.txt`, `Dockerfile`, `.env.example`.
Test locally with `uvicorn main:app --reload` and open `http://127.0.0.1:8000/docs`.

# 15. Streamlit UI (file `frontend/app.py`)
Input fields → Predict button → calls the API → shows the predicted number of orders.

# 16. Deployment
Backend: Docker image → Render web service. Frontend: Streamlit with the `API_URL` environment variable pointing to the Render URL. See `README.md`.